In [1]:
import numpy as np
import pandas as pd

### load the data

In [2]:
books = pd.read_csv("DATASET/Books.csv/Books.csv")

C:\Users\kavya\AppData\Local\Temp\ipykernel_26892\695240780.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("DATASET/Books.csv/Books.csv")


In [3]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [4]:
books.shape

(271360, 8)

In [5]:
books.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [6]:
books.dropna(inplace=True)

In [7]:
books["Year-Of-Publication"].value_counts()

Year-Of-Publication
2002    13902
2001    13714
1999    13413
2000    13373
1998    12116
        ...  
2021        1
1924        1
2012        1
1927        1
2037        1
Name: count, Length: 200, dtype: int64

In [8]:
books.duplicated().sum()

np.int64(0)

###  Create features column

In [9]:
books["features"] = (
    books["Book-Title"] + " " +
    books["Book-Author"] + " " +
    books["Publisher"] + " " +
    books["Year-Of-Publication"].astype(str)
)


### TF-IDF

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english",max_features=20000)
tfidf_matrix = tfidf.fit_transform(books["features"])

In [29]:
print(books.shape)
print(matrix.shape)

(271353, 9)
(271353, 20000)


### Train model

In [20]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6
)

model.fit(matrix)

,n_neighbors,6
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [32]:
def recommend(book_name):

    matches = books[
        books["Book-Title"].str.strip().str.lower()
        == book_name.strip().lower()
    ]

    if len(matches) == 0:
        return "Book not found"

    book_idx = matches.index.to_list()[0]

    distances, indices = model.kneighbors(
        matrix[book_idx],
        n_neighbors=6
    )

    recommendations = []

    for i in indices[0][1:]:

        recommendations.append({
            "Title": books.iloc[i]["Book-Title"],
            "Author": books.iloc[i]["Book-Author"],
            "Publisher": books.iloc[i]["Publisher"]
        })

    return recommendations

In [33]:
recommend("Classical Mythology")

[{'Title': "Who's Who in Classical Mythology (Who's Who Series)",
  'Author': 'Michael Grant',
  'Publisher': 'Oxford University Press'},
 {'Title': 'Classical mythology',
  'Author': 'Mark P. O Morford',
  'Publisher': 'Longman'},
 {'Title': 'Classical mythology',
  'Author': 'Mark P.O Morford',
  'Publisher': 'Longman'},
 {'Title': 'Classical Mythology',
  'Author': 'Mark P. O. Morford',
  'Publisher': 'John Wiley &amp; Sons'},
 {'Title': 'The Ennead',
  'Author': 'Jan Mark',
  'Publisher': 'Oxford University Press'}]

In [34]:
books = books.reset_index(drop=True)

recommend(
    books["Book-Title"].iloc[0]
)

[{'Title': "Who's Who in Classical Mythology (Who's Who Series)",
  'Author': 'Michael Grant',
  'Publisher': 'Oxford University Press'},
 {'Title': 'Classical mythology',
  'Author': 'Mark P. O Morford',
  'Publisher': 'Longman'},
 {'Title': 'Classical mythology',
  'Author': 'Mark P.O Morford',
  'Publisher': 'Longman'},
 {'Title': 'Classical Mythology',
  'Author': 'Mark P. O. Morford',
  'Publisher': 'John Wiley &amp; Sons'},
 {'Title': 'The Ennead',
  'Author': 'Jan Mark',
  'Publisher': 'Oxford University Press'}]

In [37]:
recommend("Clara Callan")
recommend("Decision in Normandy")
recommend("The Mummies of Urumchi")

[{'Title': "Shirley Barber's Count with Me!",
  'Author': 'Shirley Barber',
  'Publisher': 'The Five Mile Press'},
 {'Title': "Fear's Empire: War, Terrorism, and Democracy",
  'Author': 'Benjamin R. Barber',
  'Publisher': 'W.W. Norton &amp; Company'},
 {'Title': 'The Women Troubadours (Norton Paperback)',
  'Author': 'Magda Bogin',
  'Publisher': 'W. W. Norton &amp; Company'},
 {'Title': 'The Ladies Auxiliary: A Novel',
  'Author': 'Tova Mirvis',
  'Publisher': 'W.W. Norton &amp; Company'},
 {'Title': 'Mummies (Totally Weird)',
  'Author': 'Iqbal Hussain',
  'Publisher': 'Golden Books'}]

In [38]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(books, open("books.pkl", "wb"))
pickle.dump(matrix, open("matrix.pkl", "wb"))

In [41]:
import streamlit as st


books = pickle.load(open("books.pkl", "rb"))

selected_book = st.selectbox(
    "Select a Book",
    books["Book-Title"].unique()
)

if st.button("Recommend"):
    recs = recommend(selected_book)

    for book in recs:
        st.write(book["Title"])
        st.write(book["Author"])
        st.write("---")

2026-06-21 12:39:29.834 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-21 12:39:29.835 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-21 12:39:29.915 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-21 12:39:30.214 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-21 12:39:30.230 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-21 12:39:30.231 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-21 12:39:30.261 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-21 12:39:30.281 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [40]:
!pip install streamlit